# Packed argmax vs. number of centroids

Run top to bottom on an **A100** (Runtime > Change runtime type). At these counts it should take minutes; I haven't
timed it on a GPU.

iter06 finds the best centroid with a single `max` over packed values. It clears the low b = ceil(log2 C) bits of each
fp32 score and writes the centroid ID there. That leaves 23 − b fraction bits, so the chosen centroid's score can fall
short of the best by just under 2^(b−23) × |best score|. With unit centroids that is a cosine gap of at most
2^(b−23). At C = 2048 the bound equals the harness tolerance (2^-12) exactly, and each doubling of C doubles it.
Double-P grows C with context (`quest_attention.py` sets C ≈ tokens / 16: 2048 at 32K, 8192 at 128K).

| C | ID bits b | fraction bits kept | bound 2^(b−23) | vs. the 2^-12 tolerance |
|---|---|---|---|---|
| 2,048 | 11 | 12 | 2.44e-4 | 1× |
| 4,096 | 12 | 11 | 4.88e-4 | 2× |
| 8,192 | 13 | 10 | 9.77e-4 | 4× |

The bound is a ceiling, not a prediction. Packing can change the winner only when another centroid's score is within
one packing step of the best one. A changed winner still passes if its regret is at most 2^-12 × |key|.

**What runs.** iter06 and the **original kernel** both go through the same checker at every C: the harness's rule
(`kmeans_bench.verify` and `passes`) on every key. The original's exact argmax has no packing error, so its worst gap
is the floor set by tensor-core rounding against the fp32 reference. Anything iter06 shows above that floor comes from
packing. Sum or count failures in both kernels point to the inputs or the checker, not to packing.

**This notebook checks correctness only.** It doesn't time anything. The run time you see is compilation and the
kernels' own autotuning, not a speed comparison.

**Inputs at each C**
- **synthetic:** 32,768 random keys and C random unit centroids, made the way `kmeans_bench.make_inputs` makes them.
- **gate / heldout:** each real (layer, head) row's 32,768 keys, its 2,048 k-means centroids at IDs 0–2047, then
  C − 2048 random unit *decoy* centroids. Decoys test what happens when the ID field grows. They don't show how the
  kernel behaves with 4,096 or 8,192 real k-means centroids. That depends on how closely real centroids crowd each
  key, and these files have too few tokens to run real k-means at those counts.
- **gate shuffled / heldout shuffled:** the same centroids in a fixed random order. The packed ID decides which of two
  near-tied centroids wins, and in the unshuffled inputs the real centroids always hold the lowest IDs. Labels are
  mapped back to the unshuffled IDs to compare assignments.

**Needs**, in Google Drive under `MyDrive/kernel-agent/`: `runs/20261026-004515/iter06.py`,
`centered/real_keys_gate.pt` and `centered/real_keys_heldout.pt`.

**Predictions**, from CPU stand-ins for both kernels on the same key files. The stand-ins use the same fp32 scores as
the reference, so the original's floor reads 0 there. On the GPU it will be small but nonzero.
- **2,048:** everything passes. Worst iter06 gap: gate 1.87e-4, heldout 9.6e-5, about the same shuffled.
- **4,096:** a few real rows fail, and centroid order decides which. Gate fails unshuffled (3.56e-4) but passes
  shuffled (2.20e-4). Heldout passes unshuffled (2.40e-4) but fails shuffled (3.32e-4). Synthetic passes (8.1e-5),
  because random keys' best cosine is only about 0.3–0.5 and the packing error scales with the best score.
- **8,192:** half or more of the real rows fail in both orders (worst 7.5e-4). Synthetic passes (1.8e-4).
- **If iter06 hardcodes an 11-bit ID:** synthetic fails by a wide margin from 4,096 up (labels wrap), while the
  unshuffled real rows can still pass, because their real centroids sit below ID 2048. The shuffled rows would fail.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import hashlib, os, re, shutil
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/kernel-agent"
KERNEL_SRC = f"{D}/runs/20261026-004515/iter06.py"
GATE, HELDOUT = f"{D}/centered/real_keys_gate.pt", f"{D}/centered/real_keys_heldout.pt"
for p in [KERNEL_SRC, GATE, HELDOUT]:
    print("ok     " if os.path.exists(p) else "MISSING", p)
KERNEL = "/content/k_iter06.py"
shutil.copy(KERNEL_SRC, KERNEL)
src = open(KERNEL).read()

How iter06 sizes the ID field. If the mask comes from C (for example `(C - 1).bit_length()`), precision drops
as C grows. If it's a literal 11 or 2047, labels wrap above 2048.

In [ ]:
for i, line in enumerate(src.splitlines(), 1):
    if re.search(r"MASK|bit_length|2047|0x7ff|<<|bitcast|log2", line, re.I):
        print(f"{i:4d}  {line}")

The control: the original kernel from Double-P, the same file `kernel_explained.ipynb` writes.

In [ ]:
%%writefile k_original.py
"""The original assign kernel from Double-P (sparse_attention/index/cluster/kmeans.py, _triton_assign_kernel).

One program handles BLOCK_N keys of one batch row: score them against every centroid with tensor cores, keep the best
centroid per key, then add each key into its centroid's running sum (s) and count (cnt) with atomics.
"""

import triton
import triton.language as tl


@triton.jit
def _assign(
    K, X, S, C, M,
    stride_kz: tl.constexpr, stride_kn: tl.constexpr, stride_kd: tl.constexpr,
    stride_xz: tl.constexpr, stride_xk: tl.constexpr, stride_xd: tl.constexpr,
    stride_sz: tl.constexpr, stride_sk: tl.constexpr, stride_sd: tl.constexpr,
    stride_cz: tl.constexpr, stride_ck: tl.constexpr,
    stride_mz: tl.constexpr, stride_mn: tl.constexpr,
    num_tokens: tl.constexpr, num_centroids: tl.constexpr,
    BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, BLOCK_D: tl.constexpr,
):
    start_n = tl.program_id(0) * BLOCK_N
    batch_idx = tl.program_id(1)
    if start_n >= num_tokens:
        return
    offs_n = start_n + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    offs_d = tl.arange(0, BLOCK_D)
    n_mask = offs_n < num_tokens
    k_ptrs = K + batch_idx * stride_kz + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
    x_ptrs = X + batch_idx * stride_xz + offs_k[None, :] * stride_xk + offs_d[:, None] * stride_xd
    s_ptrs = S + batch_idx * stride_sz + offs_d[None, :] * stride_sd
    c_ptrs = C + batch_idx * stride_cz
    m_ptrs = M + batch_idx * stride_mz + offs_n * stride_mn
    k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)  # [BLOCK_N, D] keys, loaded once
    max_val = tl.zeros([BLOCK_N], dtype=tl.float32) - float("inf")
    max_idx = tl.zeros([BLOCK_N], dtype=tl.int32)
    for start_k in tl.range(0, num_centroids, BLOCK_K):
        x = tl.load(x_ptrs)  # [D, BLOCK_K] centroids
        ip = tl.dot(k, x).to(tl.float32)  # [BLOCK_N, BLOCK_K] scores on tensor cores (bf16 in, fp32 out)
        # argmax over this tile: a max, an index search and a select, every iteration
        tmp_max_val, tmp_max_idx = tl.max(ip, axis=1, return_indices=True)
        tmp_max_idx += start_k
        max_idx = tl.where(tmp_max_val > max_val, tmp_max_idx, max_idx)
        max_val = tl.maximum(tmp_max_val, max_val)
        x_ptrs += BLOCK_K * stride_xk
    tl.store(m_ptrs, max_idx, mask=n_mask)
    # s[label[n], :] += k[n, :] for the whole [BLOCK_N, D] tile in one statement. The atomics use the register layout
    # Triton chose for loading k and feeding the tensor cores, which is not arranged for atomics (see the notebook).
    tl.atomic_add(s_ptrs + max_idx[:, None] * stride_sk, k.to(tl.float32), mask=n_mask[:, None], sem="relaxed")
    # cnt[label[n]] += 1
    tl.atomic_add(c_ptrs + max_idx * stride_ck, tl.zeros_like(max_idx) + 1, mask=n_mask, sem="relaxed")


def launch(data, cent, s, cnt, m):
    """data [B, N, D] bf16, cent [B, C, D] bf16 (unit norm), s [B, C, D] fp32 and cnt [B, C] int32 zeroed, m [B, N] int32."""
    b, n, d = data.shape
    c = cent.shape[1]
    _assign[(triton.cdiv(n, 128), b, 1)](
        data, cent, s, cnt, m,
        *data.stride(), *cent.stride(), *s.stride(), *cnt.stride(), *m.stride(),
        n, c,
        BLOCK_N=128, BLOCK_K=32, BLOCK_D=d, num_warps=4, num_stages=2,  # the config kmeans.py uses
    )

The checker writes one JSON line per kernel and C. Each one runs in its own process, so a crash can't take down the
others.

In [ ]:
%%writefile c_sweep.py
"""Correctness of one assign kernel at C centroids. Prints one JSON line.

    python c_sweep.py KERNEL.py C GATE.pt HELDOUT.pt

Applies the harness's rule (kmeans_bench.verify and passes) to every key, in chunks so the [N, C] score matrix is never
materialized:
- labels in range, cnt exact, s matches the kernel's own labels (relative error < 1e-5)
- max over keys of (best score - chosen score) / |key| <= 2^-12, a cosine gap since centroids are unit norm

Inputs at C:
- synthetic: 32768 randn keys and C randn unit centroids (kmeans_bench.make_inputs, one row, seed 0)
- gate / heldout: each real (layer, head) row's 32768 keys, its 2048 k-means centroids at IDs 0-2047, then C - 2048
  random unit decoy centroids, so the ID field widens while the real centroids still compete
- gate shuffled / heldout shuffled: the same centroids in a fixed random order, since IDs decide which of two near-tied
  centroids a packed argmax keeps; labels are mapped back to the unshuffled IDs to compare assignments
"""

import importlib.util
import json
import sys
import traceback

import torch

DEV = "cuda"
D = 128
REL_TOL = 2.0 ** -12  # kmeans_bench.REL_TOL
torch.set_float32_matmul_precision("highest")  # reference scores in true fp32 (no TF32), as kmeans_bench.verify gets


def load(path):
    spec = importlib.util.spec_from_file_location("candidate", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.launch


def synthetic(c, n=32768, seed=0):
    """kmeans_bench.make_inputs(1, n, c, seed) in bf16."""
    g = torch.Generator(device=DEV).manual_seed(seed)
    data = torch.randn(1, n, D, device=DEV, generator=g).bfloat16()
    cent = torch.randn(1, c, D, device=DEV, generator=g)
    return data, (cent / cent.norm(dim=-1, keepdim=True)).bfloat16()


def with_decoys(cent, c, seed):
    """cent [2048, D] real centroids, followed by c - 2048 random unit centroids."""
    assert c >= cent.shape[0], "the sweep starts at the real centroid count"
    g = torch.Generator(device=DEV).manual_seed(seed)
    dec = torch.randn(c - cent.shape[0], D, device=DEV, generator=g)
    return torch.cat([cent, (dec / dec.norm(dim=-1, keepdim=True)).bfloat16()])


@torch.no_grad()
def verify(keys, cent, s, cnt, m, b, key_chunk=4096, cent_chunk=1 << 16):
    """kmeans_bench.verify's metrics for one row (keys [N, D], cent [C, D]), every key checked."""
    c = cent.shape[0]
    m = m.long()
    lo, hi = int(m.min()), int(m.max())
    if lo < 0 or hi >= c:
        return {"error": f"labels out of range [{lo}, {hi}]"}
    k32, c32 = keys.float(), cent.float()
    n = k32.shape[0]
    best = torch.empty(n, device=DEV)
    ref = torch.empty(n, dtype=torch.long, device=DEV)
    got = torch.empty(n, device=DEV)
    for n0 in range(0, n, key_chunk):
        kk, mm = k32[n0:n0 + key_chunk], m[n0:n0 + key_chunk]
        bv = torch.full((kk.shape[0],), float("-inf"), device=DEV)
        bi = torch.zeros(kk.shape[0], dtype=torch.long, device=DEV)
        gv = torch.full((kk.shape[0],), float("nan"), device=DEV)
        for c0 in range(0, c, cent_chunk):
            ip = kk @ c32[c0:c0 + cent_chunk].T  # [keys, centroids] fp32 scores
            v, i = ip.max(-1)
            upd = v > bv
            bv, bi = torch.where(upd, v, bv), torch.where(upd, i + c0, bi)
            rows = ((mm >= c0) & (mm < c0 + ip.shape[1])).nonzero().squeeze(1)
            gv[rows] = ip[rows, mm[rows] - c0]  # the chosen centroid's score, from the same matmul as the best
        best[n0:n0 + key_chunk], ref[n0:n0 + key_chunk], got[n0:n0 + key_chunk] = bv, bi, gv
    knorm = k32.norm(dim=-1).clamp_min(1e-30)
    regret = best - got
    _, e = torch.frexp(best)  # best = mantissa * 2^e with mantissa in [0.5, 1), so floor(log2 |best|) = e - 1
    step = torch.ldexp(torch.ones_like(best), e - 1 + b - 23)  # one packing step at the best score's exponent
    s_ref = torch.zeros(c, D, device=DEV, dtype=torch.float64).index_add_(0, m, keys.double())
    cnt_ref = torch.bincount(m, minlength=c)
    return {
        "argmax_agree": (m == ref).double().mean().item(),
        "max_regret": regret.max().item(),
        "max_rel_regret": (regret / knorm).max().item(),
        "max_regret_over_step": (regret / step).max().item(),
        "max_cos_best": (best / knorm).max().item(),
        "best_nonpositive": int((best <= 0).sum()),
        "sum_rel_err": ((s.double() - s_ref).abs().max() / s_ref.abs().max()).item(),
        "counts_exact": bool(torch.equal(cnt.long(), cnt_ref)),
    }


def run(launch, data, cent, b):
    """One checked call on data [1, N, D], cent [1, C, D], after a warmup call (compile, autotune).
    Returns the checker's metrics and the labels [N]."""
    c, n = cent.shape[1], data.shape[1]
    s = torch.zeros(1, c, D, device=DEV)
    cnt = torch.zeros(1, c, device=DEV, dtype=torch.int32)
    m = torch.full((1, n), -1, device=DEV, dtype=torch.int32)
    launch(data, cent, s, cnt, m)
    s.zero_()
    cnt.zero_()
    m.fill_(-1)
    launch(data, cent, s, cnt, m)
    torch.cuda.synchronize()
    v = verify(data[0], cent[0], s[0], cnt[0], m[0], b)
    v["passed"] = ("error" not in v and v["counts_exact"] and v["sum_rel_err"] < 1e-5
                   and v["max_rel_regret"] <= REL_TOL)
    return v, m[0].long()


def main():
    kernel, c, gate, heldout = sys.argv[1], int(sys.argv[2]), sys.argv[3], sys.argv[4]
    b = (c - 1).bit_length()  # ceil(log2 C): index bits a packed argmax needs
    out = {"kernel_path": kernel, "C": c, "index_bits": b, "fraction_bits_kept": 23 - b, "bound": 2.0 ** (b - 23),
           "rel_tol": REL_TOL, "cases": []}
    try:
        launch = load(kernel)
        data, cent = synthetic(c)
        v, m = run(launch, data, cent, b)
        if "error" not in v:
            v["ids_ge_2048"] = (m >= 2048).double().mean().item()  # about (C - 2048) / C if labels use the full ID range
        out["cases"].append({"input": "synthetic", "row": "seed 0", **v})
        del data, cent
        for name, path in [("gate", gate), ("heldout", heldout)]:
            d = torch.load(path, map_location="cpu")
            keys, cents, rows = d["keys"], d["cent"], [tuple(r) for r in d["rows"]]
            for i, (layer, head) in enumerate(rows):
                row = f"L{layer}h{head}"
                k = keys[i:i + 1].to(DEV, torch.bfloat16)
                cc = with_decoys(cents[i].to(DEV, torch.bfloat16), c, seed=100 * c + i)
                v, m0 = run(launch, k, cc[None], b)
                if "error" not in v:
                    v["ids_ge_2048"] = (m0 >= 2048).double().mean().item()  # keys won by a decoy
                out["cases"].append({"input": name, "row": row, **v})
                perm = torch.randperm(c, generator=torch.Generator().manual_seed(100 * c + i)).to(DEV)
                v, m1 = run(launch, k, cc[perm][None], b)  # position j holds centroid perm[j]
                if "error" not in v:
                    orig = perm[m1]  # back to unshuffled IDs
                    v["ids_ge_2048"] = (orig >= 2048).double().mean().item()
                    if not any(cs["input"] == name and cs["row"] == row and "error" in cs for cs in out["cases"]):
                        v["same_as_unshuffled"] = (orig == m0).double().mean().item()
                out["cases"].append({"input": f"{name} shuffled", "row": row, **v})
                del k, cc, perm
                torch.cuda.empty_cache()
    except Exception as e:  # compile errors, launch errors, illegal memory access, out of memory
        out["error"] = f"{type(e).__name__}: {e}"[:2000]
        out["traceback"] = traceback.format_exc()[-3000:]
    print(json.dumps(out))


if __name__ == "__main__":
    main()

Run the sweep. Results are saved to Drive under `c_sweep/` after every run, so a disconnect keeps whatever finished.

`COUNTS` takes any C. To look just past a bit-width boundary, add **2176** and **4224** (2048 + 128 and 4096 + 128).
Keep C a multiple of 128: the agent loop told candidates that C always is, so iter06 may load centroids without a
mask. Double-P's own C is a multiple of 128 from 32K tokens up.

In [ ]:
import json, subprocess, sys, time
import torch, triton
COUNTS = [2048, 4096, 8192]
KERNELS = {"original": "/content/k_original.py", "iter06": KERNEL}  # the control runs first at each C

sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
meta = {"kernels": {k: {"path": p, "sha256": sha(p)} for k, p in KERNELS.items()}, "iter06_source": KERNEL_SRC,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "triton": triton.__version__,
        "counts": COUNTS}
os.makedirs(f"{D}/c_sweep", exist_ok=True)
out = f"{D}/c_sweep/{time.strftime('%Y%m%d-%H%M%S')}.json"
results = []
for C in COUNTS:
    for name, path in KERNELS.items():
        t = time.time()
        p = subprocess.run([sys.executable, "c_sweep.py", path, str(C), GATE, HELDOUT], capture_output=True, text=True)
        line = next((ln for ln in reversed(p.stdout.splitlines()) if ln.startswith("{")), None)
        b = (C - 1).bit_length()
        r = json.loads(line) if line else {"C": C, "index_bits": b, "fraction_bits_kept": 23 - b,
                                           "bound": 2.0 ** (b - 23), "rel_tol": 2.0 ** -12, "cases": [],
                                           "error": f"crashed (exit {p.returncode}): {p.stderr[-1500:]}"}
        r["kernel"] = name
        results.append(r)
        with open(out, "w") as f:
            json.dump({"meta": meta, "results": results}, f, indent=1)
        print(f"C = {C:>5} {name:8s} {len(r['cases'])} rows checked in {time.time() - t:.0f}s"
              + (f"   ERROR {r['error'][:300]}" if r.get("error") else ""))
print("saved", out)

How to read the table:
- **pass:** rows that pass the harness rule: labels in range, `cnt` exact, `s` within 1e-5, and a cosine gap of at
  most 2^-12.
- **worst cos gap:** the largest (best score − chosen score) / |key| over every key in those rows. **× tol** divides it
  by 2^-12. Compare iter06 with the original at the same C: the original's value is the rounding floor.
- **regret/step:** the largest regret divided by one packing step at the best score's exponent, 2^(E + b − 23). For a
  packed argmax with b = ceil(log2 C) bits, the derivation keeps this below 1, though fp32 rounding in the dot
  products can push it a hair over. A value far above 1 means the kernel isn't behaving like that model: a different
  bit width, another bug, or a numerical assumption that doesn't hold. For the original kernel it puts rounding regret
  on the same scale.
- **IDs ≥ 2048:** on synthetic keys, the share of labels at or above 2048, which should be about (C − 2048) / C. On
  real rows, the share of keys won by a decoy.
- **same as unshuffled:** on shuffled rows, the share of keys that get the same centroid as in the unshuffled run.
  Below 1 means centroid order changed some winners. That only matters if it also pushes the gap over the tolerance.

"First observed failure" samples C at the tested values only. A failure can start anywhere between two of them, and
different inputs can start failing at different C.

In [ ]:
INPUTS = ["synthetic", "gate", "gate shuffled", "heldout", "heldout shuffled"]
nan = float("nan")
print(f"{'C':>5} {'b':>3} {'bound':>9} | {'kernel':8s} {'input':16s} {'pass':>5} {'worst cos gap':>13} {'x tol':>7} "
      f"{'regret/step':>11} {'min agree':>9} {'IDs>=2048':>9} {'same as unshuf':>14}")
first_fail = {}
for r in results:
    head = f"{r['C']:>5} {r['index_bits']:>3} {r['bound']:>9.2e}"
    if r.get("error"):
        print(f"{head} | {r['kernel']:8s} ERROR {r['error'][:200]}")
        head = " " * len(head)
    for inp in INPUTS:
        cases = [c for c in r["cases"] if c["input"] == inp]
        if not cases:
            continue
        ok = [c for c in cases if "error" not in c]
        n_pass = sum(c["passed"] for c in cases)
        worst = max((c["max_rel_regret"] for c in ok), default=nan)
        step = max((c["max_regret_over_step"] for c in ok), default=nan)
        agree = min((c["argmax_agree"] for c in ok), default=nan)
        ids = max((c.get("ids_ge_2048", nan) for c in ok), default=nan)
        same = min((c["same_as_unshuffled"] for c in ok if "same_as_unshuffled" in c), default=nan)
        other = len(ok) < len(cases) or not all(c["counts_exact"] and c["sum_rel_err"] < 1e-5 for c in ok)
        print(f"{head} | {r['kernel']:8s} {inp:16s} {n_pass:>2}/{len(cases):<2} {worst:>13.3e} {worst / r['rel_tol']:>7.2f} "
              f"{step:>11.3f} {agree:>9.5f} {ids:>9.3f} {same:>14.5f}"
              + ("   (label, sum or count errors: see below)" if other else ""))
        head = " " * len(head)
        if n_pass < len(cases):
            first_fail.setdefault((r["kernel"], inp), r["C"])
print("\nFirst observed failure among the tested C values:")
for kernel in dict.fromkeys(r["kernel"] for r in results):
    for inp in INPUTS:
        C = first_fail.get((kernel, inp))
        print(f"  {kernel:8s} {inp:16s} " + (f"C = {C}" if C else "no failure observed"))

Per row: which (layer, head) fails, plus any label, sum or count errors.

In [ ]:
for r in results:
    for inp in ["gate", "gate shuffled", "heldout", "heldout shuffled"]:
        cases = [c for c in r["cases"] if c["input"] == inp]
        if cases:
            print(f"C = {r['C']:<5} {r['kernel']:8s} {inp:16s} " + "  ".join(
                f"{c['row']} {'ok' if c['passed'] else 'FAIL'} {c.get('max_rel_regret', float('nan')):.1e}" for c in cases))
    for c in r["cases"]:
        if "error" in c or not (c["counts_exact"] and c["sum_rel_err"] < 1e-5):
            print(f"C = {r['C']} {r['kernel']} {c['input']} {c['row']}: "
                  + (c.get("error") or f"counts_exact={c['counts_exact']} sum_rel_err={c['sum_rel_err']:.1e}"))